# Урок 11. Графы: представление и обход

11 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 10](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-10.ipynb) · [Урок 12 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-12.ipynb)

---

Список смежности и матрица смежности. Обход в глубину и в ширину. Связность и компоненты. Подсчёт путей в ориентированном графе.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-11", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Задача, с которой началась теория графов

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g11/mosty.png" width="460" alt="Карта Кёнигсберга с семью мостами">

*Карта Кёнигсберга с семью мостами*

<sub>Bogdan Giuşcă, Kneiphof · CC BY-SA 3.0 · Wikimedia Commons</sub>

В Кёнигсберге XVIII века было семь мостов, соединявших два берега
и два острова. Горожане спорили: можно ли прогуляться так, чтобы
пройти по каждому мосту ровно один раз?

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g11/eyler.jpg" width="200" alt="Леонард Эйлер">

*Леонард Эйлер*

<sub>Jakob Emanuel Handmann · Public domain · Wikimedia Commons</sub>

В 1736 году Леонард Эйлер доказал, что нельзя, — и по дороге придумал
целую область математики. Его ход был такой: неважно, какой длины
мосты и какой формы острова. Важно только, что с чем соединено.
Каждый кусок суши он заменил точкой, каждый мост — линией.

Дальше рассуждение в одну строку: в любую вершину, кроме начальной
и конечной, нужно столько же раз войти, сколько выйти, — значит,
число мостов у неё должно быть чётным. В Кёнигсберге нечётное число
мостов было у всех четырёх участков суши. Маршрута не существует.

### Словарь

| Термин | Значение |
|---|---|
| вершина | объект: город, человек, страница |
| ребро | связь между двумя вершинами |
| степень вершины | сколько рёбер из неё выходит |
| ориентированный граф | рёбра со стрелками, связь односторонняя |
| взвешенный граф | у каждого ребра есть число: длина, стоимость, время |
| путь | последовательность вершин, соединённых рёбрами |
| цикл | путь, который возвращается в начало |
| связный граф | из любой вершины можно добраться до любой |

Полезное правило: сумма степеней всех вершин вдвое больше числа
рёбер — каждое ребро считается в двух вершинах.

### Два способа хранить граф

**Матрица смежности** — таблица n × n: на пересечении строки i
и столбца j стоит 1 (или вес ребра), если связь есть.

```
     А  Б  В  Г
  А  0  1  1  0
  Б  1  0  0  1
  В  1  0  0  1
  Г  0  1  1  0
```

**Список смежности** — словарь «вершина → список соседей»:

```python
{"А": ["Б", "В"], "Б": ["А", "Г"], "В": ["А", "Г"], "Г": ["Б", "В"]}
```

| | Матрица | Список |
|---|---|---|
| память | $O(n^2)$ всегда | $O(n + m)$, где m — число рёбер |
| «есть ли ребро A–B?» | мгновенно | перебор соседей |
| «перечислить соседей» | просмотр всей строки | сразу готов |
| когда удобнее | плотный граф, мало вершин | разреженный граф, много вершин |

Дороги, соцсети и ссылки в интернете — разреженные графы: у вершины
десяток соседей, а вершин миллионы. Поэтому на практике почти всегда
берут список смежности.

### Два обхода

**Обход в глубину** (DFS): идём вперёд, пока есть куда, потом
возвращаемся к последней развилке. Устроен на **стеке** — или
на рекурсии, что то же самое.

**Обход в ширину** (BFS): сначала все соседи, потом соседи соседей.
Устроен на **очереди**. Важное свойство: BFS находит путь
с наименьшим числом рёбер.

```
       А
      / \
     Б   В
    /     \
   Г       Д

  DFS от А:  А Б Г В Д        (нырнули до конца, вернулись)
  BFS от А:  А Б В Г Д        (по уровням)
```

В обоих обходах обязательна отметка «уже были»: без неё программа
зациклится на первом же цикле в графе.

## Смотрим, как это работает

### Пример 1. Граф Кёнигсберга и степени вершин

Между участками суши было по несколько мостов, поэтому храним
список соседей с повторами.

In [ ]:
кёнигсберг = {
    "А": ["Б", "Б", "В", "Г"],       # северный берег
    "Б": ["А", "А", "В", "Г", "Г"],  # остров Кнайпхоф
    "В": ["А", "Б", "Г"],            # южный берег
    "Г": ["А", "Б", "Б", "В"],       # восточный остров
}

всего_концов = 0
for вершина in кёнигсберг:
    степень = len(кёнигсберг[вершина])
    всего_концов += степень
    print(f"{вершина}: степень {степень} — {'чётная' if степень % 2 == 0 else 'нечётная'}")

print("\nМостов:", всего_концов // 2)
нечётных = sum(1 for в in кёнигсберг if len(кёнигсберг[в]) % 2 == 1)
print("Вершин с нечётной степенью:", нечётных)
print("Маршрут по всем мостам существует:", нечётных in (0, 2))

Правило Эйлера: маршрут по всем рёбрам существует, если вершин
с нечётной степенью **ноль или две**. Здесь их четыре — потому
горожане и не нашли маршрут.

### Пример 2. Матрица и список — переводим друг в друга

In [ ]:
вершины = ["А", "Б", "В", "Г"]
матрица = [
    [0, 1, 1, 0],
    [1, 0, 0, 1],
    [1, 0, 0, 1],
    [0, 1, 1, 0],
]


def в_список(матрица, имена):
    граф = {}
    for i, имя in enumerate(имена):
        граф[имя] = [имена[j] for j in range(len(имена)) if матрица[i][j]]
    return граф


граф = в_список(матрица, вершины)
print(граф)
print("Степени:", {в: len(граф[в]) for в in граф})
print("Рёбер:", sum(len(граф[в]) for в in граф) // 2)

### Пример 3. Обход в глубину

In [ ]:
дерево = {
    "А": ["Б", "В"],
    "Б": ["А", "Г"],
    "В": ["А", "Д"],
    "Г": ["Б"],
    "Д": ["В"],
}


def в_глубину(граф, старт, посещённые=None):
    if посещённые is None:
        посещённые = []
    посещённые.append(старт)
    for сосед in граф[старт]:
        if сосед not in посещённые:
            в_глубину(граф, сосед, посещённые)
    return посещённые


print("DFS:", в_глубину(дерево, "А"))

Рекурсия здесь и есть стек: каждый вложенный вызов «запоминает»,
куда вернуться. Параметр `посещённые=None` вместо `посещённые=[]` —
важная деталь: список в значении по умолчанию создаётся один раз
при определении функции и сохранялся бы между вызовами.

### Пример 4. Обход в ширину

In [ ]:
def в_ширину(граф, старт):
    посещённые = [старт]
    очередь = [старт]
    while очередь:
        текущая = очередь.pop(0)          # берём из начала — это очередь
        for сосед in граф[текущая]:
            if сосед not in посещённые:
                посещённые.append(сосед)
                очередь.append(сосед)
    return посещённые


print("BFS:", в_ширину(дерево, "А"))

Разница с DFS ровно в одной строке: `pop(0)` берёт из начала
(очередь), а `pop()` брал бы с конца (стек) — и обход превратился бы
в глубинный.

### Пример 5. Расстояния и компоненты связности

In [ ]:
def расстояния(граф, старт):
    метки = {старт: 0}
    очередь = [старт]
    while очередь:
        текущая = очередь.pop(0)
        for сосед in граф[текущая]:
            if сосед not in метки:
                метки[сосед] = метки[текущая] + 1
                очередь.append(сосед)
    return метки


print("Расстояния от А:", расстояния(дерево, "А"))

разорванный = {
    "А": ["Б"], "Б": ["А"],
    "В": ["Г"], "Г": ["В"],
    "Д": [],
}


def компонент(граф):
    увиденные = set()
    сколько = 0
    for вершина in граф:
        if вершина not in увиденные:
            сколько += 1
            for найденная in в_ширину(граф, вершина):
                увиденные.add(найденная)
    return сколько


print("Компонент связности:", компонент(разорванный))

BFS считает расстояния бесплатно: метка соседа на единицу больше
метки текущей вершины. Именно так работает поиск «на сколько
рукопожатий вы знакомы» в соцсетях.

### Пример 6. Подсчёт путей в ориентированном графе

Классика ЕГЭ: сколько существует различных путей из А в Ж?

In [ ]:
дороги = {
    "А": ["Б", "В"],
    "Б": ["Г"],
    "В": ["Г", "Д"],
    "Г": ["Е"],
    "Д": ["Е"],
    "Е": ["Ж"],
    "Ж": [],
}

пути = {в: 0 for в in дороги}
пути["А"] = 1

for вершина in ["А", "Б", "В", "Г", "Д", "Е", "Ж"]:
    for сосед in дороги[вершина]:
        пути[сосед] += пути[вершина]

print(пути)
print("Путей из А в Ж:", пути["Ж"])

Главное условие — правильный порядок обработки: вершину считают
только после всех, кто в неё ведёт. На экзаменационной схеме
такой порядок обычно виден сразу, слева направо.

## Пробуем сами

### Задача 1. Мосты Кёнигсберга

Можно ли пройти по всем семи мостам ровно по одному разу?

In [ ]:
#@title 🧩 Задача 1. Прогулка по мостам { display-mode: "form" }
#@markdown Выберите ответ
прогулка = "выбери ответ" #@param ["выбери ответ", "да", "нет"]

si.ответ("1", прогулка, "dde7950114f546d0",
         hint="Сколько вершин с нечётной степенью допускает правило Эйлера?")

### Задача 2. Степени вершин

Функция получает список смежности и возвращает словарь
«вершина → степень».

In [ ]:
def степени(граф):
    return ...

In [ ]:
si.check("2", степени, [
    ({"А": ["Б"], "Б": ["А"]}, {"А": 1, "Б": 1}),
    ({"А": ["Б", "В"], "Б": ["А"], "В": ["А"]}, {"А": 2, "Б": 1, "В": 1}),
    ({"А": []}, {"А": 0}),
])

### Задача 3. Сколько рёбер

В неориентированном графе 6 вершин, каждая соединена с каждой.
Сколько в нём рёбер?

In [ ]:
#@title 🧩 Задача 3. Полный граф { display-mode: "form" }
#@markdown Впишите число
рёбер = 0 #@param {type:"integer"}

si.ответ("3", рёбер, "e629fa6598d73276",
         hint="Каждая из шести вершин имеет степень 5, а сумма степеней вдвое больше числа рёбер.")

### Задача 4. Обход в глубину

Функция возвращает список вершин в порядке обхода в глубину,
начиная с заданной. Соседей обходите в том порядке, в котором они
записаны в списке.

In [ ]:
def дфс(граф, старт):
    return ...

In [ ]:
si.check("4", дфс, [
    (({"А": ["Б", "В"], "Б": ["А", "Г"], "В": ["А"], "Г": ["Б"]}, "А"),
     ["А", "Б", "Г", "В"]),
    (({"А": []}, "А"), ["А"]),
    (({"А": ["Б"], "Б": ["А", "В"], "В": ["Б"]}, "В"), ["В", "Б", "А"]),
])

### Задача 5. Обход в ширину

То же самое, но по уровням.

In [ ]:
def бфс(граф, старт):
    return ...

In [ ]:
si.check("5", бфс, [
    (({"А": ["Б", "В"], "Б": ["А", "Г"], "В": ["А"], "Г": ["Б"]}, "А"),
     ["А", "Б", "В", "Г"]),
    (({"А": []}, "А"), ["А"]),
])

### Задача 6. На какой структуре данных

Какой обход использует очередь?

In [ ]:
#@title 🧩 Задача 6. Очередь { display-mode: "form" }
#@markdown Выберите ответ
какой_обход = "выбери ответ" #@param ["выбери ответ", "обход в глубину", "обход в ширину"]

si.ответ("6", какой_обход, "32bef429336d075e",
         hint="Тот, который идёт по уровням.")

### Задача 7. Компоненты связности

Функция возвращает количество компонент связности графа.

In [ ]:
def компонент_связности(граф):
    return ...

In [ ]:
si.check("7", компонент_связности, [
    ({"А": ["Б"], "Б": ["А"], "В": ["Г"], "Г": ["В"], "Д": []}, 3),
    ({"А": ["Б"], "Б": ["А"]}, 1),
    ({"А": [], "Б": [], "В": []}, 3),
])

### Задача 8. Количество путей

Функция получает ориентированный граф (словарь «вершина → потомки»),
старт и финиш, возвращает количество различных путей. Вершины
обрабатывайте в алфавитном порядке.

In [ ]:
def путей(граф, старт, финиш):
    return ...

In [ ]:
si.check("8", путей, [
    (({"А": ["Б", "В"], "Б": ["Г"], "В": ["Г"], "Г": []}, "А", "Г"), 2),
    (({"А": ["Б", "В"], "Б": ["Г"], "В": ["Г", "Д"], "Г": ["Е"], "Д": ["Е"],
       "Е": ["Ж"], "Ж": []}, "А", "Ж"), 3),
    (({"А": ["Б"], "Б": []}, "А", "Б"), 1),
])

## Домашнее задание

### Домашнее задание 1. Матрица в список

Функция получает матрицу смежности и список имён вершин, возвращает
список смежности (словарь «имя → список имён соседей»).

In [ ]:
def в_список_смежности(матрица, имена):
    return ...

In [ ]:
si.check("дз1", в_список_смежности, [
    (([[0, 1], [1, 0]], ["А", "Б"]), {"А": ["Б"], "Б": ["А"]}),
    (([[0, 0], [0, 0]], ["А", "Б"]), {"А": [], "Б": []}),
    (([[0, 1, 1], [1, 0, 0], [1, 0, 0]], ["А", "Б", "В"]),
     {"А": ["Б", "В"], "Б": ["А"], "В": ["А"]}),
])

### Домашнее задание 2. Расстояние в рёбрах

Функция возвращает наименьшее количество рёбер от старта до финиша.
Если пути нет — верните `-1`.

In [ ]:
def расстояние(граф, старт, финиш):
    return ...

In [ ]:
si.check("дз2", расстояние, [
    (({"А": ["Б"], "Б": ["А", "В"], "В": ["Б"]}, "А", "В"), 2),
    (({"А": ["Б"], "Б": ["А"], "В": []}, "А", "В"), -1),
    (({"А": ["Б"], "Б": ["А"]}, "А", "А"), 0),
])

### Домашнее задание 3. Граф своего класса

Постройте граф: вершины — десять ваших одноклассников, ребро — «сидят
за одной партой хотя бы на одном уроке». Запишите его списком
смежности, посчитайте программой степени вершин, число компонент
связности и расстояние между двумя самыми далёкими знакомыми. Ответьте
письменно: похож ли получившийся граф на разреженный или на плотный
и почему.

---

### Любопытно

Теория шести рукопожатий — утверждение, что любые два человека
на Земле связаны цепочкой из шести знакомств, — проверялась
на настоящих данных. В 2011 году исследователи Facebook посчитали
расстояния в графе из 721 миллиона пользователей: среднее оказалось
4,74 рукопожатия. Считали ровно тем же обходом в ширину,
что и в примере 5, только очень аккуратно написанным.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 10](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-10.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 12 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-12.ipynb)